# VIX Decision Margins Near the Close

#### **MScFE 690 Capstone -Group 17843**

**Authors**: Edgar NAVA-edgar.nava@grovewm.com & Celestin NYANDWI-celestinnyandwi48@gmail.com

This notebook reproduces the table in Appendix C. We apply Model 2073 to synthetic SVXY −1x in one continuous simulation over Development and Validation (October 4, 2011–December 29, 2023), excluding Test.


## 1. Data and model settings

We use the same rules, costs and same-close execution assumption as `08-Model_2073_SVXY-ONEx.ipynb`. Data folders are read from `config.json`.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the project folder.")
CONFIG = json.loads((PROJECT_ROOT / "config.json").read_text(encoding="utf-8"))
RAW_DATA_DIR = PROJECT_ROOT / CONFIG["raw_data_dir"]
PROCESSED_DATA_DIR = PROJECT_ROOT / CONFIG["processed_data_dir"]

START, END = "2011-10-04", "2023-12-29"
INITIAL_CASH = 100_000.0
COMMISSION_PER_SHARE = 0.005
SLIPPAGE_PCT = 0.001
STATE_THRESHOLDS = [17.5, 22.540001, 28.805999000000003, 37.810001]
RULES = {
    "C2": {"vix_min": 23.0, "cash_fraction": 0.15,
           "target_nav": 0.70, "trailing_stop_pct": 0.20},
    "C3": {"vix_max": 28.0, "cash_fraction": 0.10,
           "invested_cash_fraction": 0.50, "trailing_stop_pct": 0.40},
    "V2": {"vix_min": 19.0},
    "V3": {"vix_max": 15.0},
}

vix = pd.read_csv(RAW_DATA_DIR / "VIX_data.csv", parse_dates=["Date"]).set_index("Date").sort_index().loc[:END]
etf = pd.read_csv(PROCESSED_DATA_DIR / "SVXY_synth-ONEx.csv", parse_dates=["Date"]).set_index("Date").sort_index().loc[START:END]
# Keep the preceding VIX observation when assigning the first session's previous state.
vix["State"] = np.searchsorted(STATE_THRESHOLDS, vix.Close, side="left") + 1
vix["Previous_state"] = vix.State.shift(1)
prices = etf[["Close"]].join(
    vix[["Close", "High", "Low", "State", "Previous_state"]].rename(columns={"Close": "VIX_Close"})
)
if (prices.empty or not vix.index.is_unique or not etf.index.is_unique
        or prices.isna().any().any() or (prices.High < prices.Low).any()):
    raise ValueError("Check the input dates and required VIX/SVXY observations.")


## 2. Continuous simulation

The account starts with USD 100,000 in cash and continues through Validation without a reset. We retain the executed orders for the comparison below.


In [2]:
def run_orders(data):
    cash, units, flag = INITIAL_CASH, 0.0, 0
    peak_close = active_stop_pct = np.nan
    orders = []
    cash_tolerance = max(1.0, INITIAL_CASH) * 1e-12

    for date, row in data.iterrows():
        price, vix_close = float(row.Close), float(row.VIX_Close)
        state, previous = int(row.State), int(row.Previous_state)
        equity_before = cash + units * price
        side, rule_id = None, ""
        target, execution_price = np.nan, np.nan
        commission, slippage_cost = 0.0, 0.0

        # Existing stops take priority over all other signals.
        if units > 0:
            peak_close = max(peak_close, price)
            if price <= peak_close * (1 - active_stop_pct):
                side, rule_id = "SELL", "V1"

        if side is None and units > 0:
            if state == 1 and previous in (2, 3, 4, 5):
                flag = 1
            if flag == 1:
                if previous == 1 and state > 1 and vix_close > RULES["V2"]["vix_min"]:
                    side, rule_id = "SELL", "V2"
                elif state == 1 and vix_close < RULES["V3"]["vix_max"]:
                    side, rule_id = "SELL", "V3"

        if side == "SELL":
            quantity = units
            execution_price = price * (1 - SLIPPAGE_PCT)
            commission = quantity * COMMISSION_PER_SHARE
            slippage_cost = quantity * price * SLIPPAGE_PCT
            cash += quantity * execution_price - commission
            units, flag = 0.0, 0
            peak_close = active_stop_pct = np.nan
        else:
            if previous == 2 and state == 3 and vix_close > RULES["C2"]["vix_min"]:
                rule_id = "C2"
                target = RULES[rule_id]["target_nav"] if units > 0 else RULES[rule_id]["cash_fraction"]
            elif previous == 4 and state == 3 and vix_close < RULES["C3"]["vix_max"]:
                rule_id = "C3"
                target = RULES[rule_id]["invested_cash_fraction"] if units > 0 else RULES[rule_id]["cash_fraction"]

            if np.isfinite(target):
                execution_price = price * (1 + SLIPPAGE_PCT)
                all_in_price = execution_price + COMMISSION_PER_SHARE
                if units == 0 or rule_id == "C3":
                    quantity = target * cash / all_in_price
                else:
                    # C2 targets position value / NAV after transaction costs.
                    gap = max(0.0, target * equity_before - units * price)
                    quantity = min(cash / all_in_price,
                        gap / (price + target * (price * SLIPPAGE_PCT + COMMISSION_PER_SHARE)))

                if quantity * all_in_price > cash_tolerance:
                    side = "BUY"
                    commission = quantity * COMMISSION_PER_SHARE
                    slippage_cost = quantity * price * SLIPPAGE_PCT
                    cash -= quantity * execution_price + commission
                    units += quantity
                    active_stop_pct = RULES[rule_id]["trailing_stop_pct"]
                    peak_close = price

        # Remove floating-point residue without allowing borrowing.
        if -cash_tolerance <= cash < 0:
            cash = 0.0
        if cash < 0 or units < 0:
            raise ArithmeticError(f"Negative cash or units at {date}.")
        if units == 0:
            flag = 0

        if side is not None:
            orders.append({
                "Date": date, "Side": side, "Rule ID": rule_id,
                "Close": price, "Execution price": execution_price, "Units": quantity,
                "Commission ($)": commission, "Slippage ($)": slippage_cost,
                "Cash after": cash, "Units after": units,
            })

    order_columns = ["Date", "Side", "Rule ID", "Close", "Execution price", "Units",
                     "Commission ($)", "Slippage ($)", "Cash after", "Units after"]
    return pd.DataFrame(orders, columns=order_columns)

orders = run_orders(prices)


## 3. VIX decision margins

The margin is the distance from the VIX Close to the nearest boundary of the executed rule, keeping the previous state and portfolio situation fixed. We compare each margin with that same day's VIX High–Low range divided by 26, the number of 15-minute intervals in a 6.5-hour session.

The table includes executed VIX-based purchases and sales. V1 trailing-stop sales depend on the ETF price and are excluded. Each trade here is one purchase or sale, not a round trip.


In [3]:
trades = orders[orders["Rule ID"].isin(["C2", "C3", "V2", "V3"])].set_index("Date").join(
    prices[["VIX_Close", "High", "Low"]]
)
close = trades.VIX_Close
b2, b3 = STATE_THRESHOLDS[1:3]
margins = {
    "C2": np.minimum(close - RULES["C2"]["vix_min"], b3 - close),
    "C3": np.minimum(close - b2, RULES["C3"]["vix_max"] - close),
    "V2": close - RULES["V2"]["vix_min"],
    "V3": RULES["V3"]["vix_max"] - close,
}
for rule, values in margins.items():
    mask = trades["Rule ID"].eq(rule)
    trades.loc[mask, "Margin"] = values.loc[mask]
trades["Reference"] = (trades.High - trades.Low) / 26
trades["At or below reference"] = trades.Margin <= trades.Reference

def summarize(group):
    return [len(group), group.Margin.mean(), group.Reference.mean(),
            int(group["At or below reference"].sum())]

labels = {"C2": "C2 purchase", "C3": "C3 purchase", "V2": "V2 sale", "V3": "V3 sale"}
rows = {label: summarize(trades[trades["Rule ID"].eq(rule)]) for rule, label in labels.items()}
rows["Total"] = summarize(trades)
table = pd.DataFrame.from_dict(rows, orient="index", columns=[
    "Trades", "Mean margin (VIX points)", "Mean 15-minute reference¹ (VIX points)",
    "Trades with margin at or below their own reference",
]).rename_axis("Rule")


In [4]:
styled = (table.style
    .format({table.columns[0]: "{:.0f}", table.columns[1]: "{:.3f}",
             table.columns[2]: "{:.3f}", table.columns[3]: "{:.0f}"})
    .set_uuid("vix_margins")
    .set_caption("VIX Decision Margins — Development and Validation")
    .set_table_styles([
        {"selector": "caption", "props": [("text-align", "left"), ("color", "#255c80"),
             ("font-size", "16px"), ("padding", "8px 0")]},
        {"selector": "th", "props": [("background-color", "#255c80"), ("color", "white"),
             ("text-align", "left"), ("padding", "9px 12px")]},
        {"selector": "td", "props": [("padding", "9px 12px"), ("text-align", "right")]},
        {"selector": "tbody tr:nth-child(even)", "props": [("background-color", "#edf3f7")]},
        {"selector": "tbody tr:last-child", "props": [("font-weight", "bold")]},
        {"selector": "table", "props": [("border-collapse", "collapse")]},
    ])
)
display({"text/html": styled.to_html(), "text/plain": table.to_string(float_format="{:.3f}".format)}, raw=True)


,Trades,Mean margin (VIX points),Mean 15-minute reference¹ (VIX points),Trades with margin at or below their own reference
Rule,,,,
C2 purchase,23,1.151,0.163,0
C3 purchase,24,0.967,0.149,5
V2 sale,5,0.958,0.142,0
V3 sale,9,0.236,0.085,3
Total,61,0.928,0.144,8


¹ The 15-minute reference is an arithmetic division of the daily range, not an observed intraday movement.

Of 67 executed purchases and sales, 61 depended on VIX conditions. Eight of these 61 trades (13.1%) had a margin at or below their own reference. This descriptive comparison does not establish whether earlier signals would change decisions or performance.
